# Guiana Zarr data check

1. **Full-area preview** — right after load, whole grid per time step (u / v / ssh).
2. **Full domain + windows** — same full grid with random training crop boxes overlaid.
3. **Random crops** — zoomed patches (what the model actually sees during training).

Set `GUIANA_ZARR_PATH` (environment variable or in the config cell). AWS credentials required for `s3://` paths.

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# margem_equatorial_tensor.zarr stores bathymetry as an ocean_tensor channel, not a separate array
from src.data.dataloaders.guiana_margem_tensor import GuianaDataloader
from src.data.useful_transforms import random_crop


In [5]:
ZARR_PATH = os.getenv(
    "GUIANA_ZARR_PATH",
    "s3://oceangen-data/margem_equatorial_tensor.zarr",
)

DT = 24 #*3
CONDITIONING_STEPS = 4
FORECAST_STEPS = 56 #3
CROP_SIZE = 128

START_TIME = "2016-01-01"
END_TIME = "2019-12-31"

# START_TIME = "2020-01-01"
# END_TIME = "2021-12-31"

N_SAMPLES_TO_PLOT = 3
SEED = 42

# Full-domain section: how many random CROP_SIZE windows to draw on each map
N_RANDOM_WINDOWS = 8


In [6]:
base_kwargs = dict(
    path=ZARR_PATH,
    dt=DT,
    conditioning_steps=CONDITIONING_STEPS,
    forecast_steps=FORECAST_STEPS,
)
if START_TIME is not None:
    base_kwargs["start_time"] = START_TIME
if END_TIME is not None:
    base_kwargs["end_time"] = END_TIME

print("Opening Zarr:", ZARR_PATH)
dataset_full = GuianaDataloader(**base_kwargs)
dataset = GuianaDataloader(**base_kwargs, transform=random_crop(CROP_SIZE))
print("Valid samples:", len(dataset))

info = dataset_full.info()
print("Raw time interval:", info["time"]["raw_interval"])
print("Valid sample interval:", info["time"]["valid_interval"])
print("Native dt:", info["time"]["native_dt"])
print("Sample dt:", info["time"]["sample_dt"])
print("Grid (H, W):", info["space"]["height"], info["space"]["width"])
print("Channels:", info["channels"]["loaded_names"])
print(f"Crop {CROP_SIZE}x{CROP_SIZE} fits in grid:", info["space"]["height"] >= CROP_SIZE and info["space"]["width"] >= CROP_SIZE)


Opening Zarr: s3://oceangen-data/margem_equatorial_tensor.zarr


Valid samples: 35064
Raw time interval: ['2013-01-01 00:00:00', '2023-12-31 00:00:00']
Valid sample interval: ['2016-01-01 00:00:00', '2020-02-28 23:00:00']
Native dt: 0 days 01:00:00
Sample dt: 1 days 00:00:00
Grid (H, W): 431 547
Channels: ['u', 'v', 'ssh']
Crop 128x128 fits in grid: True


## Full-area preview

Right after opening the Zarr: one sample, **full grid** (no crop), all time steps × u / v / ssh.

In [12]:
preview_idx = int(PREVIEW_IDX) if PREVIEW_IDX is not None else int(
    np.random.default_rng(SEED).choice(len(dataset_full))
)

times = dataset_full.sample_times(preview_idx)
x, y_meta, mask = dataset_full[preview_idx]

n_steps = dataset_full.total_steps

n_ch = 3
cond = y_meta["y"].detach().cpu().numpy().reshape(
    dataset_full.conditioning_steps, n_ch, *y_meta["y"].shape[-2:]
)
fcst = x.detach().cpu().numpy().reshape(
    dataset_full.forecast_steps, n_ch, *x.shape[-2:]
)
seq = np.concatenate([cond, fcst], axis=0)

print(seq.shape)

np.save("data/twoMonth.npy",seq)

(60, 3, 431, 547)


In [ ]:
return
PREVIEW_IDX = None  # set e.g. 0 or 11468 to fix sample; None = random with SEED

if len(dataset_full) == 0:
    raise RuntimeError("dataset_full is empty — check ZARR_PATH, credentials, and date range.")

preview_idx = int(PREVIEW_IDX) if PREVIEW_IDX is not None else int(
    np.random.default_rng(SEED).choice(len(dataset_full))
)
times = dataset_full.sample_times(preview_idx)
x, y_meta, mask = dataset_full[preview_idx]

n_steps = dataset_full.total_steps
n_ch = 3
cond = y_meta["y"].detach().cpu().numpy().reshape(
    dataset_full.conditioning_steps, n_ch, *y_meta["y"].shape[-2:]
)
fcst = x.detach().cpu().numpy().reshape(
    dataset_full.forecast_steps, n_ch, *x.shape[-2:]
)
seq = np.concatenate([cond, fcst], axis=0)
H_FULL, W_FULL = seq.shape[-2], seq.shape[-1]
t_labels = [str(t)[:16] for t in times]
channel_names = ["u", "v", "ssh"]

vmin, vmax = np.nanpercentile(seq, 2), np.nanpercentile(seq, 98)

fig, axes = plt.subplots(n_steps, 3, figsize=(12, 2.8 * n_steps), squeeze=False)
fig.suptitle(
    f"Full area | idx={preview_idx} | {H_FULL}x{W_FULL} | start {t_labels[0]}",
    y=1.01,
)
for t in range(n_steps):
    for c in range(3):
        ax = axes[t, c]
        im = ax.imshow(seq[t, c], cmap="RdBu_r", vmin=vmin, vmax=vmax)
        if t == 0:
            ax.set_title(channel_names[c])
        if c == 0:
            ax.set_ylabel(t_labels[t], rotation=0, labelpad=52, fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
plt.tight_layout()
plt.show()

print(f"preview_idx={preview_idx}, times={[str(t) for t in times]}")

: 

In [8]:
def stack_time_channels(flat: torch.Tensor, n_frames: int, n_channels: int = 3) -> np.ndarray:
    t = flat.detach().cpu().numpy()
    return t.reshape(n_frames, n_channels, t.shape[-2], t.shape[-1])


def load_sequence(dataset, idx: int):
    x, y, mask = dataset[idx]
    cond = stack_time_channels(y["y"], dataset.conditioning_steps)
    fcst = stack_time_channels(x, dataset.forecast_steps)
    seq = np.concatenate([cond, fcst], axis=0)
    times = dataset.sample_times(idx)
    return seq, mask.detach().cpu().numpy(), times, y


def load_full_sequence(dataset, idx: int):
    """Same as load_sequence but without spatial crop (full H x W)."""
    return load_sequence(dataset, idx)


def sample_random_crop_boxes(h: int, w: int, crop_h: int, crop_w: int, n: int, seed=None):
    """Return list of (top, left) corners — same sampling as random_crop()."""
    rng = np.random.default_rng(seed)
    tops = rng.integers(0, h - crop_h + 1, size=n)
    lefts = rng.integers(0, w - crop_w + 1, size=n)
    return list(zip(tops.tolist(), lefts.tolist()))


def crop_coverage_map(h: int, w: int, crop_h: int, crop_w: int, boxes):
    """Count how many sampled windows cover each pixel (union coverage diagnostic)."""
    cov = np.zeros((h, w), dtype=np.int32)
    for top, left in boxes:
        cov[top : top + crop_h, left : left + crop_w] += 1
    return cov


In [9]:
N_SAMPLES_TO_GEN = 50

rng = np.random.default_rng(SEED)
indices = rng.choice(len(dataset), size=min(N_SAMPLES_TO_GEN, len(dataset)), replace=False)

data_sims   = np.zeros((N_SAMPLES_TO_GEN, dataset.total_steps, 3, CROP_SIZE, CROP_SIZE), dtype=np.float32)
data_mask   = np.zeros((N_SAMPLES_TO_GEN, CROP_SIZE, CROP_SIZE), dtype=np.float32)
data_day    = np.zeros((N_SAMPLES_TO_GEN,), dtype=np.int16)
data_hour   = np.zeros((N_SAMPLES_TO_GEN,), dtype=np.int8)
data_latlon = np.zeros((N_SAMPLES_TO_GEN, 2), dtype=np.float32)

for sample_i, idx in enumerate(indices):
    isGood = False
    while not isGood:
        seq, mask, times, y = load_sequence(dataset, int(idx))
        if float(mask.mean())<0.4:
            isGood = True
    
    t_labels = [str(t)[:16] for t in times]

    dt = datetime.strptime(t_labels[3], "%Y-%m-%d %H:%M")
    day_of_year = dt.timetuple().tm_yday  # 239
    hour = dt.hour

    data_sims[sample_i,:,:,:,:] = seq
    data_mask[sample_i,:,:] = mask[0,:,:]
    data_day[sample_i] = day_of_year
    data_hour[sample_i] = hour
    data_latlon[sample_i] = y['latlon'].tolist()

    print(sample_i,day_of_year, hour,t_labels[0])

np.save('data/sampleY_sims.npy', data_sims)
np.save('data/sampleY_mask.npy', data_mask)
np.save('data/sampleY_day.npy', data_day)
np.save('data/sampleY_hour.npy', data_hour)
np.save('data/sampleY_latlon.npy', data_latlon)

KeyboardInterrupt: 

In [ ]:
rng = np.random.default_rng(SEED)
if len(dataset) == 0:
    raise RuntimeError("Dataset length is 0. Check ZARR_PATH, credentials, and date range.")

indices = rng.choice(len(dataset), size=min(N_SAMPLES_TO_PLOT, len(dataset)), replace=False)
print("Plotting indices:", indices.tolist())

CHANNEL_NAMES = ["u", "v", "ssh"]
n_steps = dataset.total_steps

for sample_i, idx in enumerate(indices):
    seq, mask, times, y = load_sequence(dataset, int(idx))
    t_labels = [str(t)[:16] for t in times]

    fig, axes = plt.subplots(n_steps, 3, figsize=(10, 2.6 * n_steps), squeeze=False)
    fig.suptitle(
        f"Sample {sample_i} | idx={idx} | start={t_labels[0]} | crop {seq.shape[-2]}x{seq.shape[-1]}",
        y=1.01,
    )

    vmin = np.nanpercentile(seq, 2)
    vmax = np.nanpercentile(seq, 98)

    for t in range(n_steps):
        for c in range(3):
            ax = axes[t, c]
            im = ax.imshow(seq[t, c], cmap="RdBu_r", vmin=vmin, vmax=vmax)
            if t == 0:
                ax.set_title(CHANNEL_NAMES[c])
            if c == 0:
                ax.set_ylabel(t_labels[t], rotation=0, labelpad=48, fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])

    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
    plt.tight_layout()
    plt.show()

    fig2, ax2 = plt.subplots(1, 1, figsize=(4, 4))
    ax2.imshow(mask[0], cmap="gray")
    ax2.set_title(f"Invalid mask (last conditioning) | idx={idx}")
    ax2.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"  lat/lon center: {y['latlon'].tolist()}")
    print(f"  dates (last frame): {y['dates'].tolist()}")
    print(f"  mask fraction invalid: {float(mask.mean()):.4f}")


: 

In [ ]:
from matplotlib.patches import Rectangle

if len(dataset_full) == 0:
    raise RuntimeError("Dataset length is 0. Check ZARR_PATH, credentials, and date range.")

rng_full = np.random.default_rng(SEED)
full_indices = rng_full.choice(
    len(dataset_full), size=min(N_SAMPLES_TO_PLOT, len(dataset_full)), replace=False
)
print("Full-domain sample indices:", full_indices.tolist())

H_FULL = info["space"]["height"]
W_FULL = info["space"]["width"]
crop_h = crop_w = CROP_SIZE
CHANNEL_NAMES = ["u", "v", "ssh"]
n_steps = dataset_full.total_steps
window_colors = plt.cm.tab10(np.linspace(0, 1, N_RANDOM_WINDOWS))

for sample_i, idx in enumerate(full_indices):
    seq, mask, times, y = load_full_sequence(dataset_full, int(idx))
    t_labels = [str(t)[:16] for t in times]
    boxes = sample_random_crop_boxes(H_FULL, W_FULL, crop_h, crop_w, N_RANDOM_WINDOWS, seed=SEED + int(idx))
    coverage = crop_coverage_map(H_FULL, W_FULL, crop_h, crop_w, boxes)
    covered_frac = (coverage > 0).mean()

    vmin = np.nanpercentile(seq, 2)
    vmax = np.nanpercentile(seq, 98)

    # Fields + window overlays (rows = time, cols = channel)
    fig, axes = plt.subplots(n_steps, 3, figsize=(12, 2.8 * n_steps), squeeze=False)
    fig.suptitle(
        f"Full domain | sample {sample_i} | idx={idx} | start={t_labels[0]} | "
        f"{H_FULL}x{W_FULL} | {N_RANDOM_WINDOWS} random {CROP_SIZE}x{CROP_SIZE} windows",
        y=1.01,
    )

    for t in range(n_steps):
        for c in range(3):
            ax = axes[t, c]
            im = ax.imshow(seq[t, c], cmap="RdBu_r", vmin=vmin, vmax=vmax)
            for k, (top, left) in enumerate(boxes):
                ax.add_patch(
                    Rectangle(
                        (left, top),
                        crop_w,
                        crop_h,
                        fill=False,
                        edgecolor=window_colors[k],
                        linewidth=1.2,
                    )
                )
            if t == 0:
                ax.set_title(CHANNEL_NAMES[c])
            if c == 0:
                ax.set_ylabel(t_labels[t], rotation=0, labelpad=52, fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])

    fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.02, pad=0.02)
    plt.tight_layout()
    plt.show()

    # Coverage from this draw of N windows (not full training run)
    fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4))
    axes2[0].imshow(coverage, cmap="viridis")
    axes2[0].set_title(f"Pixel hit count ({N_RANDOM_WINDOWS} windows)")
    axes2[0].axis("off")
    axes2[1].imshow(mask[0], cmap="gray")
    axes2[1].set_title("Invalid mask (last conditioning)")
    axes2[1].axis("off")
    fig2.suptitle(
        f"idx={idx} | pixels touched by ≥1 window: {covered_frac:.1%} of grid",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

    print(f"  Example windows (top, left): {boxes[:3]}{'...' if len(boxes) > 3 else ''}")
    print(f"  Union coverage ({N_RANDOM_WINDOWS} draws): {covered_frac:.1%} of {H_FULL}x{W_FULL}")

: 

In [ ]:
idx = int(indices[0])
a, _, _, _ = load_sequence(dataset, idx)
b, _, _, _ = load_sequence(dataset, idx)
print(f"Same index {idx} twice — identical crop? {np.allclose(a, b)} (expect False)")


Same index 11468 twice — identical crop? False (expect False)


In [ ]:
from joblib import Parallel, delayed
import numpy as np
from datetime import datetime

def process_sample(idx):
    """Process a single sample — runs in parallel."""
    isGood = False
    while not isGood:
        seq, mask, times, y = load_sequence(dataset, int(idx))
        if float(mask.mean()) < 0.4:
            isGood = True

    t_labels = [str(t)[:16] for t in times]
    dt = datetime.strptime(t_labels[3], "%Y-%m-%d %H:%M")
    day_of_year = dt.timetuple().tm_yday
    hour = dt.hour
    latlon = y['latlon'].tolist()

    return seq, mask[0, :, :], day_of_year, hour, latlon


N_SAMPLES_TO_GEN = 5000
rng = np.random.default_rng(SEED)
indices = rng.choice(len(dataset), size=min(N_SAMPLES_TO_GEN, len(dataset)), replace=False)

# Run in parallel — adjust n_jobs to your CPU count (-1 = use all cores)
results = Parallel(n_jobs=30, verbose=10)(
    delayed(process_sample)(idx) for idx in indices
)

# Unpack results
data_sims   = np.zeros((N_SAMPLES_TO_GEN, dataset.total_steps, 3, CROP_SIZE, CROP_SIZE), dtype=np.float32)
data_mask   = np.zeros((N_SAMPLES_TO_GEN, CROP_SIZE, CROP_SIZE), dtype=np.float32)
data_day    = np.zeros((N_SAMPLES_TO_GEN,), dtype=np.int16)
data_hour   = np.zeros((N_SAMPLES_TO_GEN,), dtype=np.int8)
data_latlon = np.zeros((N_SAMPLES_TO_GEN, 2), dtype=np.float32)

for i, (seq, mask, day, hour, latlon) in enumerate(results):
    data_sims[i]   = seq
    data_mask[i]   = mask
    data_day[i]    = day
    data_hour[i]   = hour
    data_latlon[i] = latlon

np.save('data/sample_16_19_sims.npy', data_sims)
np.save('data/sample_16_19_mask.npy', data_mask)
np.save('data/sample_16_19_day.npy', data_day)
np.save('data/sample_16_19_hour.npy', data_hour)
np.save('data/sample_16_19_latlon.npy', data_latlon)

[Parallel(n_jobs=30)]: Using backend LokyBackend with 30 concurrent workers.
[Parallel(n_jobs=30)]: Done   1 tasks      | elapsed:    7.6s
[Parallel(n_jobs=30)]: Done  12 tasks      | elapsed:   15.2s
[Parallel(n_jobs=30)]: Done  25 tasks      | elapsed:   17.8s
[Parallel(n_jobs=30)]: Done  38 tasks      | elapsed:   19.3s
[Parallel(n_jobs=30)]: Done  53 tasks      | elapsed:   21.8s
[Parallel(n_jobs=30)]: Done  68 tasks      | elapsed:   24.0s
[Parallel(n_jobs=30)]: Done  85 tasks      | elapsed:   27.5s
[Parallel(n_jobs=30)]: Done 102 tasks      | elapsed:   30.9s
[Parallel(n_jobs=30)]: Done 121 tasks      | elapsed:   34.4s
[Parallel(n_jobs=30)]: Done 140 tasks      | elapsed:   37.5s
[Parallel(n_jobs=30)]: Done 161 tasks      | elapsed:   43.0s
[Parallel(n_jobs=30)]: Done 182 tasks      | elapsed:   46.3s
[Parallel(n_jobs=30)]: Done 205 tasks      | elapsed:   51.2s
[Parallel(n_jobs=30)]: Done 228 tasks      | elapsed:   55.3s
[Parallel(n_jobs=30)]: Done 253 tasks      | elapsed:  

## Checks

- `len(dataset) > 0` and sensible `info()` time range
- Fields evolve across rows (time)
- Mask mostly low; high values = missing data
- Random crop: same index usually gives different patches

If OK, use the same `GuianaDataloader` + `random_crop` in your training code.